In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from unidecode import unidecode
import plotly.graph_objects as go
import json
from pathlib import Path
import webbrowser
import joblib

# =================================================
# PATH
# =================================================
CSV_PATH = "../outputs/prediksi_2024.csv"
SHP_PATH = "../maps/BATAS KABUPATEN KOTA DESEMBER 2019 DUKCAPIL.shp"
OUT_HTML = "../outputs/dashboard_peta_perbandingan_prediksi.html"
MODEL_PATH = "../models/lasso_model.pkl"

In [3]:
# =================================================
# Normalisasi nama wilayah
# =================================================
def norm(s):
    if pd.isna(s): return s
    s = unidecode(str(s)).upper().strip()
    s = s.replace("-", " ").replace("/", " ")
    for w in ["KABUPATEN ", "KOTA ", "KAB. ", "ADM. ", "DAERAH ", "ISTIMEWA "]:
        s = s.replace(w, "")
    return " ".join(s.split())

In [3]:
# =================================================
# Load CSV
# =================================================
print("[INFO] Load CSV...")
df = pd.read_csv(CSV_PATH)

req = {"Wilayah", "P1"}
if not req.issubset(df.columns):
    raise ValueError(f"CSV harus punya kolom {req}")

# =================================================
# Load Model
# =================================================
print("[INFO] Loading model...")
model = joblib.load(MODEL_PATH)
print("[OK] Model loaded")

# =================================================
# Feature columns
# =================================================
feature_cols = ["UHH", "HLS", "RLS", "Pengeluaran", "TPT", "Kepadatan"]
missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"CSV tidak memiliki fitur: {missing}")

# =================================================
# Prediksi LASSO
# =================================================
print("[INFO] Predicting...")
df["prediksi"] = model.predict(df[feature_cols])
print("[OK] Prediction done")

df["join_key"] = df["Wilayah"].apply(norm)

[INFO] Load CSV...
[INFO] Loading model...
[OK] Model loaded
[INFO] Predicting...
[OK] Prediction done


In [4]:
# =================================================
# Load Shapefile
# =================================================
print("[INFO] Load Shapefile...")
gdf = gpd.read_file(SHP_PATH)

# =================================================
# Cari kolom nama wilayah otomatis
# =================================================
print("[INFO] Mencari kolom nama wilayah...")

csv_keys = set(df["join_key"])
obj_cols = [c for c in gdf.columns if gdf[c].dtype == "object"]

best_col, best_score = None, 0
for c in obj_cols:
    cand = set(gdf[c].astype(str).apply(norm))
    score = len(cand & csv_keys)
    if score > best_score:
        best_col, best_score = c, score

if best_col is None:
    raise ValueError("Tidak bisa menemukan kolom nama wilayah di shapefile")

name_col = best_col
print(f"[OK] Kolom wilayah terdeteksi: {name_col}")

gdf["join_key"] = gdf[name_col].apply(norm)

# =================================================
# Filter hanya wilayah yang ada di CSV (Jawa Tengah)
# =================================================
gdf = gdf[gdf["join_key"].isin(csv_keys)].copy()
print("[OK] Polygon tersaring:", len(gdf))

# =================================================
# Merge
# =================================================
merged = gdf.merge(df[["join_key", "P1", "prediksi"]], on="join_key", how="left")

# Hilangkan duplikat multipart polygon
merged = merged.drop_duplicates(subset=["join_key"])

[INFO] Load Shapefile...
[INFO] Mencari kolom nama wilayah...
[OK] Kolom wilayah terdeteksi: KAB_KOTA
[OK] Polygon tersaring: 35


In [5]:
# =================================================
# CRS ke WGS84
# =================================================
try:
    if merged.crs is None or merged.crs.to_epsg() != 4326:
        merged = merged.to_crs(4326)
except:
    pass

# =================================================
# Simplify geometry (kecilkan HTML)
# =================================================
print("[INFO] Simplifying geometry...")
merged["geometry"] = merged["geometry"].simplify(
    tolerance=0.005,
    preserve_topology=True
)

# =================================================
# Hitung ERROR
# =================================================
merged["error"] = (merged["P1"] - merged["prediksi"]).abs()

# ----------------------------------------------
# Relative Error (%)
# ----------------------------------------------
merged["rel_error"] = np.where(
    merged["P1"] != 0,
    (merged["error"] / merged["P1"]) * 100,
    np.nan
)


[INFO] Simplifying geometry...


In [6]:
# =================================================
# Statistik
# =================================================
mae = float(merged["error"].mean())
rmse = float(np.sqrt(((merged["P1"] - merged["prediksi"])**2).mean()))

mean_p1 = float(merged["P1"].mean())
max_p1_val = float(merged["P1"].max())
min_p1_val = float(merged["P1"].min())
max_p1_row = merged.loc[merged["P1"].idxmax()]
min_p1_row = merged.loc[merged["P1"].idxmin()]

mean_pred = float(merged["prediksi"].mean())
max_pred_val = float(merged["prediksi"].max())
min_pred_val = float(merged["prediksi"].min())
max_pred_row = merged.loc[merged["prediksi"].idxmax()]
min_pred_row = merged.loc[merged["prediksi"].idxmin()]

mean_err = float(merged["error"].mean())
max_err_val = float(merged["error"].max())
min_err_val = float(merged["error"].min())
max_err_row = merged.loc[merged["error"].idxmax()]
min_err_row = merged.loc[merged["error"].idxmin()]

mean_rel = float(np.nanmean(merged["rel_error"]))
max_rel_val = float(np.nanmax(merged["rel_error"]))
min_rel_val = float(np.nanmin(merged["rel_error"]))

max_rel_row = merged.loc[merged["rel_error"].idxmax()]
min_rel_row = merged.loc[merged["rel_error"].idxmin()]


# =================================================
# Warna
# =================================================
vmin = float(min(merged["P1"].min(), merged["prediksi"].min()))
vmax = float(max(merged["P1"].max(), merged["prediksi"].max()))

err_min = float(merged["error"].min())
err_max = float(merged["error"].max())

rel_min = float(np.nanmin(merged["rel_error"]))
rel_max = float(np.nanmax(merged["rel_error"]))

# =================================================
# Titik aman untuk label wilayah (bukan centroid)
# =================================================
label_points = merged.geometry.representative_point()
merged["lon"] = label_points.x
merged["lat"] = label_points.y

# =================================================
# HAPUS OBJECT GEOMETRY TAMBAHAN SEBELUM JSON
# =================================================
json_ready = merged.drop(columns=[c for c in ["centroid"] if c in merged.columns])

# =================================================
# GeoJSON
# =================================================
geojson = json.loads(json_ready.to_json())

locations = merged["join_key"]
custom = np.array(merged[name_col].fillna("-"))

In [7]:
# ----------------------------------------------
# Statistik P1
# ----------------------------------------------
mean_p1 = float(merged["P1"].mean())
max_p1_val = float(merged["P1"].max())
min_p1_val = float(merged["P1"].min())
max_p1_row = merged.loc[merged["P1"].idxmax()]
min_p1_row = merged.loc[merged["P1"].idxmin()]

# ----------------------------------------------
# Statistik Prediksi
# ----------------------------------------------
mean_pred = float(merged["prediksi"].mean())
max_pred_val = float(merged["prediksi"].max())
min_pred_val = float(merged["prediksi"].min())
max_pred_row = merged.loc[merged["prediksi"].idxmax()]
min_pred_row = merged.loc[merged["prediksi"].idxmin()]

# ----------------------------------------------
# Statistik Error
# ----------------------------------------------
mean_err = float(merged["error"].mean())
max_err_val = float(merged["error"].max())
min_err_val = float(merged["error"].min())
max_err_row = merged.loc[merged["error"].idxmax()]
min_err_row = merged.loc[merged["error"].idxmin()]

# ----------------------------------------------
# Relative Error (%)
# ----------------------------------------------
merged["rel_error"] = np.where(
    merged["P1"] != 0,
    (merged["error"] / merged["P1"]) * 100,
    np.nan
)

# ----------------------------------------------
# Range warna
# ----------------------------------------------
vmin = float(min(merged["P1"].min(), merged["prediksi"].min()))
vmax = float(max(merged["P1"].max(), merged["prediksi"].max()))

err_min = float(merged["error"].min())
err_max = float(merged["error"].max())

# ----------------------------------------------
# GeoJSON
# ----------------------------------------------
geo_cols = [c for c in merged.columns if c != "centroid"]
geojson = json.loads(merged[geo_cols].to_json())

# ----------------------------------------------
# Data dasar
# ----------------------------------------------
locations = merged["join_key"]
custom = np.array(merged[name_col].fillna("-"))

In [8]:
# =================================================
# Hover
# =================================================
hover_p1 = (
    "<b>%{customdata[0]}</b><br>"
    "P1: %{z:.2f}%<br>"
    f"Rata-rata: {mean_p1:.2f}%"
)

hover_pred = (
    "<b>%{customdata[0]}</b><br>"
    "Prediksi: %{z:.2f}%<br>"
    f"Rata-rata: {mean_pred:.2f}%"
)

hover_err = (
    "<b>%{customdata[0]}</b><br>"
    "Error: %{z:.2f}%<br>"
    f"Rata-rata Error: {mean_err:.2f}%"
)

hover_rel = (
    "<b>%{customdata[0]}</b><br>"
    "Relative Error: %{z:.2f}%<br>"
    f"Rata-rata: {mean_rel:.2f}%"
)


In [9]:
# =================================================
# Trace Choropleth
# =================================================
trace_p1 = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["P1"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="P1 (%)",
    hovertemplate=hover_p1,
    customdata=np.c_[custom],
    visible=True
)

trace_pred = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["prediksi"],
    colorscale="OrRd",
    zmin=vmin, zmax=vmax,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="Prediksi (%)",
    hovertemplate=hover_pred,
    customdata=np.c_[custom],
    visible=False
)

trace_err = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["error"],
    colorscale="Reds",
    zmin=err_min, zmax=err_max,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="Error (%)",
    hovertemplate=hover_err,
    customdata=np.c_[custom],
    visible=False
)

trace_rel = go.Choropleth(
    geojson=geojson,
    featureidkey="properties.join_key",
    locations=locations,
    z=merged["rel_error"],
    colorscale="Purples",
    zmin=rel_min, 
    zmax=rel_max,
    marker_line_color="white",
    marker_line_width=0.5,
    colorbar_title="Rel Error (%)",
    hovertemplate=hover_rel,
    customdata=np.c_[custom],
    visible=False
)

# =================================================
# Trace Label Wilayah
# =================================================
trace_labels = go.Scattergeo(
    lon=merged["lon"],
    lat=merged["lat"],
    text=merged[name_col],
    mode="text",
    textfont=dict(size=9, color="black"),
    hoverinfo="skip",
    showlegend=False,
    visible=True
)

# =================================================
# Subtitle
# =================================================
title_base = "Indeks Kedalaman Kemiskinan"

subtitle_p1 = (
    f"P1 • Mean: {mean_p1:.2f}% • "
    f"Max: {max_p1_row[name_col]} ({max_p1_val:.2f}%) • "
    f"Min: {min_p1_row[name_col]} ({min_p1_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)

subtitle_pred = (
    f"Prediksi • Mean: {mean_pred:.2f}% • "
    f"Max: {max_pred_row[name_col]} ({max_pred_val:.2f}%) • "
    f"Min: {min_pred_row[name_col]} ({min_pred_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)

subtitle_err = (
    f"Error • Mean: {mean_err:.2f}% • "
    f"Max: {max_err_row[name_col]} ({max_err_val:.2f}%) • "
    f"Min: {min_err_row[name_col]} ({min_err_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)

subtitle_rel = (
    f"Rel Error • Mean: {mean_rel:.2f}% • "
    f"Max: {max_rel_row[name_col]} ({max_rel_val:.2f}%) • "
    f"Min: {min_rel_row[name_col]} ({min_rel_val:.2f}%) • "
    f"MAE: {mae:.3f} • RMSE: {rmse:.3f}"
)


In [11]:
# =================================================
# Buat FIGURE dulu
# =================================================
fig = go.Figure(data=[trace_p1, trace_pred, trace_err, trace_rel, trace_labels])

fig.update_layout(
    title=dict(
        text=f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>",
        x=0.5
    ),
    margin=dict(l=40, r=40, t=80, b=40),
    geo=dict(
        fitbounds="locations",
        visible=False,
        projection_type="mercator",
        center=dict(lat=-7.2, lon=110.0),
        lataxis=dict(range=[-9.0, -5.5]),
        lonaxis=dict(range=[108.5, 112.0])
    ),
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="P1 (Aktual)",
                    method="update",
                    args=[
                        {"visible":[True, False, False, False, True]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_p1}</span>"}}
                    ]
                ),
                dict(
                    label="Prediksi (LASSO)",
                    method="update",
                    args=[
                        {"visible":[False, True, False, False, True]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_pred}</span>"}}
                    ]
                ),
                dict(
                    label="Error (Absolut)",
                    method="update",
                    args=[
                        {"visible":[False, False, True, False, True]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_err}</span>"}}
                    ]
                ),
                dict(
                    label="Relative Error (%)",
                    method="update",
                    args=[
                        {"visible":[False, False, False, True, True]},
                        {"title":{"text":f"<b>{title_base}</b><br><span style='font-size:12px'>{subtitle_rel}</span>"}}
                    ]
                )
            ],
            direction="down",
            x=0.05, xanchor="left",
            y=0.95, yanchor="top"
        )
    ],
    annotations=[
        dict(
            x=0.5, y=-0.05, xref="paper", yref="paper",
            text="Sumber: BPS Jawa Tengah (2024) • Model: LASSO Regression",
            showarrow=False,
            font=dict(size=11, color="#555")
        )
    ]
)


# =================================================
# Save HTML
# =================================================
fig.write_html(
    OUT_HTML,
    include_plotlyjs="cdn",
    full_html=True
)

print(f"[OK] Dashboard saved: {OUT_HTML}")
webbrowser.open(Path(OUT_HTML).resolve().as_uri())

[OK] Dashboard saved: ../outputs/dashboard_peta_perbandingan_prediksi.html


True

In [4]:
import geopandas as gpd
import matplotlib.pyplot as plt
import os

# --- 1. PERSIAPAN DATA ---
# Pastikan df_prediksi sudah ada (dari langkah sebelumnya)
# df_prediksi harus punya kolom: ['Wilayah', 'Aktual_2024', 'Prediksi_2024', 'Error']

# Load Shapefile (Sesuaikan path jika perlu)
path_maps = "../maps/"  # Sesuaikan dengan folder kamu
filename_shp = "BATAS KABUPATEN KOTA DESEMBER 2019 DUKCAPIL.shp"
shapefile_path = os.path.join(path_maps, filename_shp)

gdf = gpd.read_file(shapefile_path)

# Fungsi Standarisasi Nama (Sama seperti kodemu sebelumnya)
def format_nama_peta(nama):
    nama = str(nama).upper().strip()
    if nama.startswith("KAB.") or nama.startswith("KAB "):
        nama = nama.replace("KAB.", "").replace("KAB ", "").strip()
        return "Kabupaten " + nama.title()
    elif nama.startswith("KOTA"):
        return nama.title()
    else:
        return "Kabupaten " + nama.title()

# Terapkan format nama
gdf['Wilayah_Join'] = gdf['KAB_KOTA'].apply(format_nama_peta)
df_prediksi['Wilayah_Join'] = df_prediksi['Wilayah'] # Asumsi nama di df_prediksi sudah bersih

# Merge Data Prediksi ke Peta
gdf_evaluasi = gdf.merge(df_prediksi, on='Wilayah_Join', how='inner')

# --- 2. KONFIGURASI PLOTTING ---
# Tentukan Range Warna agar Peta Aktual & Prediksi punya standar yang SAMA ("Apel to Apel")
vmin_p1 = min(gdf_evaluasi['Aktual_2024'].min(), gdf_evaluasi['Prediksi_2024'].min())
vmax_p1 = max(gdf_evaluasi['Aktual_2024'].max(), gdf_evaluasi['Prediksi_2024'].max())

# Tentukan Range Error agar 0 berwarna putih (tengah-tengah)
max_abs_err = max(abs(gdf_evaluasi['Error'].min()), abs(gdf_evaluasi['Error'].max()))

# --- 3. BUAT 3 PETA SEKALIGUS ---
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# PETA 1: AKTUAL 2024
gdf_evaluasi.plot(
    column='Aktual_2024',
    ax=axes[0],
    cmap='OrRd',      # Merah untuk kemiskinan
    vmin=vmin_p1,     # Kunci skala min
    vmax=vmax_p1,     # Kunci skala max
    legend=True,
    legend_kwds={'label': "Tingkat Kemiskinan (%)", 'orientation': "horizontal", 'shrink': 0.8}
)
axes[0].set_title("A. Kondisi Aktual 2024\n(Data Real)", fontsize=14, fontweight='bold')
axes[0].axis('off')

# PETA 2: PREDIKSI MODEL
gdf_evaluasi.plot(
    column='Prediksi_2024',
    ax=axes[1],
    cmap='OrRd',      # Warna SAMA dengan Aktual
    vmin=vmin_p1,     # Skala SAMA dengan Aktual
    vmax=vmax_p1,
    legend=True,
    legend_kwds={'label': "Prediksi P1 (%)", 'orientation': "horizontal", 'shrink': 0.8}
)
axes[1].set_title("B. Prediksi Model Lasso\n(Estimasi)", fontsize=14, fontweight='bold')
axes[1].axis('off')

# PETA 3: SEBARAN ERROR (RESIDUAL)
gdf_evaluasi.plot(
    column='Error',
    ax=axes[2],
    cmap='coolwarm',  # Biru (Negatif) - Putih (Nol) - Merah (Positif)
    vmin=-max_abs_err, # Paksa range simetris biar 0 = Putih
    vmax=max_abs_err,
    legend=True,
    legend_kwds={'label': "Error (Aktual - Prediksi)", 'orientation': "horizontal", 'shrink': 0.8}
)
axes[2].set_title("C. Peta Sebaran Error\n(Biru=Over, Merah=Under)", fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()

# Simpan Gambar
output_file = "../visualisasi_EDA/Peta_Evaluasi_Model_2024.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Peta Evaluasi berhasil disimpan di: {output_file}")

NameError: name 'df_prediksi' is not defined